# Static Multi-Asset Portfolio (Crypto)

Compare static long-only portfolio optimizers on 5–7 crypto pairs.
Business logic lives in `src/crypto_hf/` — this notebook only runs the pipeline and displays reports.

In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

from crypto_hf.config import load_static_portfolio_config
from crypto_hf.pipeline.static_multi_asset_portfolio import run_static_multi_asset_portfolio_pipeline

pd.options.display.float_format = "{:.4f}".format

## 1. Config and symbols

In [ ]:
config = load_static_portfolio_config(PROJECT_ROOT / "configs" / "static_multi_asset_portfolio.yaml")
config.symbols

## 2. Run pipeline

In [ ]:
outputs = run_static_multi_asset_portfolio_pipeline(
    config,
    reports_dir=PROJECT_ROOT / "reports",
)
print(f"Aligned rows: {outputs.alignment_rows}")
print(f"Train: {len(outputs.train_returns)} | Test: {len(outputs.test_returns)}")

## 3. Correlation and weights

In [ ]:
outputs.correlation_matrix

In [ ]:
outputs.weights

## 4. Portfolio metrics and diagnostics

In [ ]:
outputs.metrics

In [ ]:
outputs.diagnostics

## 5. Figures

In [ ]:
from IPython.display import Image, display

figures_dir = PROJECT_ROOT / "reports" / "figures"
for name in [
    "static_portfolio_equity_curves.png",
    "static_portfolio_drawdowns.png",
    "static_portfolio_weights.png",
    "static_portfolio_correlation_heatmap.png",
    "static_portfolio_risk_return_scatter.png",
]:
    path = figures_dir / name
    if path.exists():
        display(Image(filename=path))

## 6. Interpretation

- **equal_weight** — simple baseline; no estimation risk.
- **inverse_volatility** — naive risk parity; down-weights volatile coins.
- **min_variance** — risk-focused optimizer using train covariance.
- **max_sharpe** — return/risk trade-off using train expected returns and covariance.
- **hrp** — hierarchical risk parity; robust diversification via clustering.

Weights are fixed at the start of the **test period** (static buy-and-hold, no rebalancing).
Negative test performance is possible and does not indicate a pipeline bug.

Agents, dynamic rebalancing, and live trading are out of scope for this stage.